# Iceberg — User Journey

Iceberg wraps the Parquet data in a metadata layer. A lightweight SQLite catalog tracks tables/namespaces; the actual data files live on S3.

**Steps:** create → query → filtered query → append → evolve schema

These map to the planned `ln.integrations.iceberg` helpers (`create_iceberg`, `query_iceberg`, `append_dataset`, `evolve_schema`).

In [1]:
import time
from contextlib import contextmanager

timings = {}

@contextmanager
def bench(step):
    """Time a step and record it for the final benchmark table."""
    t0 = time.perf_counter()
    yield
    timings[step] = time.perf_counter() - t0
    print(f"  {step}: {timings[step]:.3f}s")

## 1. Create — load LaminDB data into an Iceberg table

`collection.open()` streams the shards from S3 as one PyArrow table, which is then written into the Iceberg table. (Future: `ln.integrations.iceberg.create_iceberg(collection)`.)

In [4]:
import lamindb as ln
import pyarrow as pa
from pyiceberg.catalog import load_catalog
from pyiceberg.expressions import EqualTo, And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.types import BooleanType
import os
catalog_dir = os.getcwd()

# 1. Track lineage (letting LaminDB generate unique UID)
ln.track("pAeQCxxy3ZfF", project="Lakehouse benchmarks v1")
collection = ln.Collection.get("K6X8Ejk3fjgAZT6h0000")

# 2. Access: Load data from LaminDB (S3) into PyArrow for conversion
with bench("load_data"):
    arrow_table = collection.open().to_table()

# 3. Configure S3-native catalog
# The SageMaker environment automatically provides credentials
catalog = load_catalog(
    "s3_catalog",
    **{
        "type": "sql",
        "uri": f"sqlite:///{catalog_dir}/iceberg_catalog.db",
        "warehouse": "s3://lamindata/iceberg_warehouse",
        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO"
    }
)

# 4. Create: Write the collection to an S3-backed Iceberg table
with bench("create"):
    if not catalog.namespace_exists("genomics"):
        catalog.create_namespace("genomics")
    table = catalog.create_table_if_not_exists("genomics.cnv_vcf", schema=arrow_table.schema)
    table.overwrite(arrow_table)

print(f"Total rows: {len(table.scan().to_arrow()):,}")

→ loaded Transform('pAeQCxxy3ZfF0000', key='iceberg_pipeline.ipynb'), re-started Run('p3Xxxe2ot5LbUR4u') at 2026-06-13 14:02:17 UTC
→ notebook imports: lamindb-core==2.5.1 pandas==2.3.3 pyarrow==24.0.0 pyiceberg==0.11.1
  load_data: 11.280s


OSError: When initiating multiple part upload for key 'iceberg_warehouse/genomics/cnv_vcf/metadata/00000-0b4e972b-29d3-44ef-ac87-f12dc084d797.metadata.json' in bucket 'lamindata': AWS Error ACCESS_DENIED during CreateMultipartUpload operation: Anonymous users cannot initiate multipart uploads.  Please authenticate. (Request ID: F6QNFQ2CTTGJ4Z52)

## 2. Query — per-sample stats and recurrent regions

In [ ]:
def calculate_sample_stats(arrow_table):
    """Summary statistics per sample."""
    df = arrow_table.to_pandas()
    rows = []
    for name in df["SAMPLE_NAME"].unique():
        s = df[df["SAMPLE_NAME"] == name]
        dels = s[s["INFO_SVLEN"] < 0]
        rows.append({
            "Sample": name,
            "Total_CNVs": len(s),
            "Deletions": len(dels),
            "Median_Deletion_Size": abs(dels["INFO_SVLEN"].median()) if not dels.empty else 0,
            "Homozygous_CNVs": (s["SAMPLE_GT"] == "1/1").sum(),
            "Heterozygous_CNVs": (s["SAMPLE_GT"] == "0/1").sum(),
        })
    return pd.DataFrame(rows)

def identify_recurrent_regions(arrow_table, proximity=1000):
    """Regions with recurrent CNVs across >=2 samples."""
    df = arrow_table.select(["CHROM", "POS", "SAMPLE_NAME"]).to_pandas()
    df["region_key"] = df["CHROM"] + ":" + ((df["POS"] // proximity) * proximity).astype(str)
    counts = df.groupby("region_key")["SAMPLE_NAME"].nunique()
    return counts[counts >= 2]

In [ ]:
with bench("query_stats"):
    full = table.scan().to_arrow()
    stats_df = calculate_sample_stats(full)
stats_df.head()

In [ ]:
with bench("query_recurrent"):
    recurrent = identify_recurrent_regions(full)
print(f"Identified {len(recurrent)} recurrent regions.")

## 3. Filtered query — row filter pushed down at scan time

In [ ]:
from pyiceberg.expressions import EqualTo, And, GreaterThanOrEqual, LessThanOrEqual

with bench("filtered_query"):
    filtered = table.scan(
        row_filter=And(
            EqualTo("CHROM", "1"),
            GreaterThanOrEqual("POS", 1_000_000),
            LessThanOrEqual("POS", 50_000_000),
        )
    ).to_arrow()
print(f"Variants in chr1:1M-50M: {len(filtered)}")

## 4. Append — atomic, snapshot-isolated

Concurrent readers always see a consistent state. (Future: `ln.integrations.iceberg.append_dataset(table, new_arrow)`.)

In [ ]:
with bench("append"):
    table.append(arrow_table.slice(0, 10))  # placeholder — replace with real new data
print(f"Snapshots: {len(table.history())} | Rows: {len(table.scan().to_arrow()):,}")

## 5. Evolve schema — add a column without rewriting Parquet files

(Future: `ln.integrations.iceberg.evolve_schema(table, ...)`.)

In [ ]:
from pyiceberg.types import BooleanType

with bench("evolve_schema"):
    with table.update_schema() as update:
        update.add_column("QC_PASS", BooleanType())
print("Schema updated with QC_PASS")

## Benchmark summary

In [ ]:
import pandas as pd
pd.DataFrame(
    [{"step": k, "seconds": round(v, 3)} for k, v in timings.items()]
)

In [ ]:
try:
    ln.finish()
except Exception:
    pass
